# Avant le TP3 : faire tourner des consommateurs dans le notebook

La version Markdown du TP3 ouvre trois terminaux pour faire tourner plusieurs consommateurs d'un même groupe. Dans le notebook, chaque consommateur est une **instance** qui tourne en arrière-plan. Cette démonstration montre comment on les lance, les observe et les arrête, sur un topic jetable `demo-tp3` de 4 partitions :

1. une instance seule reçoit toutes les partitions ;
2. une seconde instance déclenche un rééquilibrage ;
3. le groupe se décrit en Python comme en ligne de commande ;
4. une instance arrêtée proprement rend ses partitions aussitôt.

Chaque étape montre la version **terminal**, puis la version **Python**. Une étape qui ne fait que lire exécute les deux. Une étape qui modifie le cluster n'en exécute qu'une : l'autre est affichée à côté, **à ne pas lancer en plus**.

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab.

Les noms et les valeurs sont définis une seule fois, ici. Dans un terminal de JupyterLab, définissez d'abord les mêmes variables : `export TOPIC=demo-tp3 GROUP=demo-tp3`.

Création du topic, équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic $TOPIC --partitions 4 --replication-factor 3
```

**En Python**

In [ ]:
import os
import threading
import time

from confluent_kafka import Consumer, KafkaError, KafkaException
from confluent_kafka.admin import AdminClient, NewTopic

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}

TOPIC = "demo-tp3"
GROUP = "groupe-demo-tp3"
PARTITIONS = 4
RF = 3
MAX_DELAY = 60  # secondes d'attente au plus pour un rééquilibrage

admin = AdminClient(CONF)
try:
    admin.create_topics([NewTopic(TOPIC, num_partitions=PARTITIONS, replication_factor=RF)])[TOPIC].result()
    print(f"Le topic {TOPIC} a été créé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.TOPIC_ALREADY_EXISTS:
        raise
    print(f"Le topic {TOPIC} existe déjà")

: 

## La classe `Instance`

Elle fonctionne comme celle du TP3. Un `Consumer` abonné au topic tourne dans un fil d'exécution : sa boucle `consume()` fait vivre son appartenance au groupe, et `partitions` donne ses partitions du moment.

En terminal, une instance est un consommateur lancé dans son propre terminal, qui tourne jusqu'à `Ctrl+C` (**ne pas lancer en plus**) :

```bash
docker exec -it kafka-tools kafka-console-consumer.sh --bootstrap-server $BS --topic $TOPIC --group $GROUP
```

In [2]:
class Instance:
    """Une instance d'un service consommateur : un Consumer qui tourne dans un fil d'exécution.

    Attributs à lire :
      group      -- group.id de l'instance
      partitions -- partitions qu'elle détient en ce moment (liste triée), vide
                    tant que le premier rééquilibrage n'a pas abouti
      consumer   -- le Consumer confluent-kafka sous-jacent
      launched   -- attribut de classe : les instances lancées depuis la dernière
                    exécution de cette cellule
    """

    launched = []

    def __init__(self, group, topic="orders", **config):
        """Crée le Consumer, l'abonne au topic et démarre sa boucle en arrière-plan.

        group    -- group.id : les instances d'un même groupe se partagent les partitions
        topic    -- topic auquel l'instance s'abonne
        **config -- réglages du Consumer qui s'ajoutent aux valeurs par défaut ou les
                    remplacent, par exemple **{"group.protocol": "consumer"}

        Le retour est immédiat, avant toute affectation : attendre ensuite les
        partitions, par exemple wait_for(lambda: x.partitions, "...").
        """
        self.group = group
        self.partitions = []
        self._stop_event = threading.Event()
        self.consumer = Consumer({**CONF, "group.id": group,
                                  "auto.offset.reset": "earliest", "log_level": 4, **config})
        self.consumer.subscribe([topic])
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        Instance.launched.append(self)

    def _loop(self):
        """Corps du fil d'exécution : consomme et met à jour partitions jusqu'à l'arrêt.

        Les messages lus sont ignorés ; leurs offsets sont validés automatiquement
        (enable.auto.commit, actif par défaut). À la sortie, demandée par stop()
        ou causée par une exception, close() fait quitter le groupe proprement.
        """
        try:
            while not self._stop_event.is_set():
                # consume() fait aussi vivre l'appartenance au groupe (rééquilibrages, validations)
                self.consumer.consume(100, timeout=0.5)
                self.partitions = sorted(tp.partition for tp in self.consumer.assignment())
        finally:
            # Départ propre : le coordinateur est prévenu et rééquilibre aussitôt
            self.consumer.close()

    def stop(self):
        """Arrête l'instance proprement, comme Ctrl+C sur un consommateur en terminal.

        La boucle se termine et close() prévient le coordinateur, qui rééquilibre
        aussitôt les partitions libérées. Attend la fin du fil 15 s au plus.
        """
        self._stop_event.set()
        self._thread.join(15)

    @classmethod
    def stop_all(cls):
        """Arrête toutes les instances de Instance.launched, puis vide la liste.

        À utiliser en cas de doute, par exemple après avoir exécuté deux fois une
        cellule qui crée des instances. Une instance lancée avant la dernière
        exécution de la cellule de la classe n'est plus dans la liste.
        """
        for instance in cls.launched:
            instance.stop()
        cls.launched = []

    def __repr__(self):
        """Affichage court, par exemple : Instance(demo-tp3, partitions [0, 1])."""
        return "Instance(%s, partitions %s)" % (self.group, self.partitions)


def wait_for(condition, message):
    """Rappelle la condition chaque seconde ; lève une erreur si elle reste fausse trop longtemps."""
    for _ in range(MAX_DELAY):
        if condition():
            return
        time.sleep(1)
    raise TimeoutError(message)

## 1. Une instance seule

**En Python** : la fonction `wait_for()` rappelle la condition chaque seconde, jusqu'à ce qu'elle soit vraie, et s'arrête avec une erreur au bout de `MAX_DELAY` secondes.

In [3]:
x = Instance(GROUP, topic=TOPIC)
wait_for(lambda: x.partitions, "x n'a reçu aucune partition")
print(x)

Instance(demo-tp3, partitions [0, 1, 2, 3])


## 2. Une seconde instance : rééquilibrage

**En Python** : on attend que les deux instances aient chacune des partitions.

In [4]:
y = Instance(GROUP, topic=TOPIC)
wait_for(lambda: x.partitions and y.partitions, "le rééquilibrage n'a pas abouti")
print(x)
print(y)

Instance(demo-tp3, partitions [2, 3])
Instance(demo-tp3, partitions [0, 1])


## 3. Décrire le groupe

**En terminal** : l'outil Java donne l'état, la stratégie d'affectation et le nombre de membres.

In [5]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group $GROUP --state


GROUP           COORDINATOR (ID)          ASSIGNMENT-STRATEGY  STATE                #MEMBERS
demo-tp3        kafka-2:19092  (2)        range                Stable               2


**En Python** : `describe_consumer_groups()` renvoie la même vue, avec les partitions de chaque membre.

In [6]:
admin = AdminClient(CONF)
group = admin.describe_consumer_groups([GROUP])[GROUP].result()
print(group.state.name, group.type.name, group.partition_assignor)
for m in group.members:
    print(m.member_id, sorted(tp.partition for tp in m.assignment.topic_partitions))

STABLE CLASSIC range
rdkafka-f8a7c43c-12fd-4c72-9ba4-5a65f1b66213 [2, 3]
rdkafka-4c657014-3d69-469a-9490-adc9a5435f91 [0, 1]


## 4. Arrêt propre d'une instance

En terminal, c'est `Ctrl+C` dans le terminal du consommateur : il prévient le coordinateur en partant.

**En Python**

In [7]:
y.stop()
wait_for(lambda: len(x.partitions) == PARTITIONS, "x n'a pas récupéré toutes les partitions")
print(x)

Instance(demo-tp3, partitions [0, 1, 2, 3])


## Ménage

`Instance.stop_all()` arrête toutes les instances lancées : à utiliser en cas de doute, par exemple après avoir exécuté deux fois une cellule qui en crée. Le groupe et le topic de démonstration sont ensuite supprimés.

Équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --delete --group $GROUP
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --delete --topic $TOPIC
```

**En Python** : un groupe ou un topic déjà absent n'est pas une erreur.

In [8]:
Instance.stop_all()
admin = AdminClient(CONF)

try:
    admin.delete_consumer_groups([GROUP])[GROUP].result()
except KafkaException as e:
    if e.args[0].code() != KafkaError.GROUP_ID_NOT_FOUND:
        raise
    print(f"groupe {GROUP} absent")

try:
    admin.delete_topics([TOPIC])[TOPIC].result()
except KafkaException as e:
    if e.args[0].code() != KafkaError.UNKNOWN_TOPIC_OR_PART:
        raise
    print(f"topic {TOPIC} absent")

Les groupes restants. **En terminal**

In [9]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list

console-consumer-49793
loadgen-consumers
console-consumer-25073


**En Python**

In [10]:
admin = AdminClient(CONF)
print("groupes :", sorted(g.group_id for g in admin.list_consumer_groups().result().valid))

groupes : ['console-consumer-25073', 'console-consumer-49793', 'loadgen-consumers']


## La méthode du TP3

Ouvrez `tps/jour1/tp3_groupes_retention/enonce.ipynb`.

- Les consommateurs du TP sont des instances : lancez-les, attendez leur affectation avec une boucle d'attente, arrêtez-les avec `stop()`.
- Une cellule qui crée des instances ne s'exécute qu'une fois ; en cas de doute, `Instance.stop_all()`.
- Les étapes de rétention et de compaction attendent le travail du broker : une à deux minutes par étape, pendant lesquelles la cellule reste en cours d'exécution (`[*]`).